# Project scaffold — Applied Generative AI

**SDAIA Academy · this is the file you build your project on.**

Twelve sections. The structure is written; the `TODO` markers are yours.
By Thursday, `demo()` at the bottom should run end to end in front of the room.

| Section | What it is |
|---|---|
| 1 | Config — key, model, constants |
| 2 | Ingest — point it at **your** documents |
| 3 | Chunk — your Day 2 chunker |
| 4 | Index — embed and store |
| 5 | Retrieve — hybrid search, given complete |
| 6 | Tools — one example plus a slot for yours |
| 7 | Agent loop — given complete |
| 8 | Guardrails — one example plus a slot |
| 9 | Instrumentation — the logging from Day 4 |
| 10 | `demo()` — what you run on Thursday |
| 11 | Golden set — five questions that prove it works |
| 12 | README template — copy into your repository |

**Rubric reminder:** architecture choice and justification is worth 25 of the
100 points. Being able to say *why* you chose this shape, in one sentence, is
worth more than any extra feature.

## 1 · Config

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# Installs are quiet (-q) so the output stays readable on a projector.
!pip install -q google-genai chromadb rank-bm25 numpy pandas

import os, json, time                    # standard library
from google import genai                 # the SDK
from google.genai import types           # config and content types
import numpy as np
import pandas as pd
import chromadb
from rank_bm25 import BM25Okapi
from datetime import datetime

# ── Your API key ──────────────────────────────────────────────────────
# The key lives in Colab Secrets, never in the notebook. If this cell
# fails, that is almost always why.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    if not API_KEY:
        raise ValueError("empty")
except Exception:
    raise SystemExit(
        "\n" + "=" * 68 +
        "\nNo API key found.\n"
        "\n  1. Click the KEY icon in the left sidebar of Colab."
        "\n  2. Click 'Add new secret'."
        "\n  3. Name it exactly:  GEMINI_API_KEY"
        "\n  4. Paste your key from aistudio.google.com"
        "\n  5. Turn ON 'Notebook access' for this notebook."
        "\n  6. Run this cell again."
        "\n" + "=" * 68
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash-lite"          # fast and cheap; what we use all week
EMBED_MODEL = "gemini-embedding-001"    # free tier, which is what makes Day 2 possible

print("Ready. Model:", MODEL)

# ── Project constants ─────────────────────────────────────────────────
PROJECT_NAME = "TODO: name your project"      # ← TODO
CHUNK_SIZE = 500        # tokens; tune it against your golden set
CHUNK_OVERLAP = 50
TOP_K = 4
MAX_STEPS = 5           # agent step cap. Never remove this.

## 2 · Ingest

Point this at **your own** documents. Upload them to Colab (the folder icon in
the left sidebar) or mount Google Drive.

Keep `source` and `page` from the very first step. Without them you cannot
cite, and citations are 25 of your 100 rubric points via "visibly grounded
answers".

In [ ]:
def load_my_documents(folder="my_docs"):
    """Return a list of {"text": str, "source": str, "page": int}."""
    docs = []

    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)

        if name.endswith(".txt"):
            with open(path, encoding="utf-8") as fh:
                docs.append({"text": fh.read(), "source": name, "page": 1})

        elif name.endswith(".pdf"):
            # ← TODO (2 lines): extract text per page.
            #   pip install pypdf, then loop over reader.pages and append
            #   one dict per page so `page` is accurate for citations.
            pass

        elif name.lower().endswith((".png", ".jpg", ".jpeg")):
            # Scanned page: the Day 2 vision route.
            page_bytes = open(path, "rb").read()
            r = client.models.generate_content(
                model=MODEL,
                contents=[types.Part.from_bytes(data=page_bytes, mime_type="image/png"),
                          "Transcribe this page exactly. Do not summarise."])
            docs.append({"text": r.text, "source": name, "page": 1})

    return docs


os.makedirs("my_docs", exist_ok=True)
docs = load_my_documents()
print(len(docs), "documents loaded")

## 3 · Chunk

Your Day 2 chunker. Structure before size: paragraphs first, then length.

In [ ]:
def chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""

    for para in paras:
        if len(current) + len(para) < size * 4:      # ~4 chars per token
            current += para + "\n\n"
        else:
            # ← TODO (2 lines): append the finished chunk, then restart
            #   `current` from its last overlap*4 characters plus this para.
            pass

    if current.strip():
        chunks.append(current.strip())
    return chunks


records = [{"text": ch, "source": d["source"], "page": d["page"]}
           for d in docs for ch in chunk(d["text"])]

print(len(records), "chunks")

## 4 · Index

Embed and store. The batching and the sleep are there because the free tier rate-limits.

In [ ]:
def embed(texts, task="RETRIEVAL_DOCUMENT", batch=32, pause=1.0):
    out = []
    for i in range(0, len(texts), batch):
        r = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts[i:i + batch],
            config=types.EmbedContentConfig(task_type=task))
        out += [e.values for e in r.embeddings]
        time.sleep(pause)
    return out


vectors = embed([r["text"] for r in records])

db = chromadb.Client()
try:
    db.delete_collection(PROJECT_NAME)       # guard against re-runs
except Exception:
    pass
col = db.create_collection(PROJECT_NAME)

col.add(ids=[f"c{i}" for i in range(len(records))],
        documents=[r["text"] for r in records],
        embeddings=vectors,
        metadatas=[{"source": r["source"], "page": r["page"]} for r in records])

DOC_MATRIX = np.array(vectors, dtype=float)
bm25 = BM25Okapi([r["text"].lower().split() for r in records])
print("Indexed", col.count(), "chunks")

## 5 · Retrieve

**Given complete.** Hybrid search: keyword and vector, combined. Tune `alpha` against your golden set in section 11, not by feel.

In [ ]:
def norm(x):
    x = np.array(x, dtype=float)
    return (x - x.min()) / (np.ptp(x) + 1e-9)


def hybrid_search(query, k=TOP_K, alpha=0.5):
    kw = norm(bm25.get_scores(query.lower().split()))
    qv = np.array(embed([query], task="RETRIEVAL_QUERY", pause=0)[0], dtype=float)
    dots = DOC_MATRIX @ qv
    vec = norm(dots / (np.linalg.norm(DOC_MATRIX, axis=1) * np.linalg.norm(qv) + 1e-9))

    score = alpha * vec + (1 - alpha) * kw
    return [records[i] for i in np.argsort(score)[::-1][:k]]


GROUNDED = """You are {name}. Answer using ONLY the reference material
between the tags. Cite source and page for every fact. If the material does
not contain the answer, say you do not know.

<reference>
{context}
</reference>

Question: {question}"""


def answer(question, k=TOP_K):
    hits = hybrid_search(question, k)
    context = "\n\n".join(f"[{h['source']} p.{h['page']}]\n{h['text']}" for h in hits)
    r = client.models.generate_content(
        model=MODEL,
        contents=GROUNDED.format(name=PROJECT_NAME, context=context, question=question),
        config=types.GenerateContentConfig(temperature=0.1))
    return r.text

## 6 · Tools

One worked example, then a slot for yours. Remember: the `description` is what the model reads when deciding. A vague description is a broken tool.

In [ ]:
def search_documents(query: str) -> str:
    """Your retriever, as a tool."""
    hits = hybrid_search(query, k=3)
    return json.dumps([{"text": h["text"][:600], "source": h["source"],
                        "page": h["page"]} for h in hits])


search_decl = types.FunctionDeclaration(
    name="search_documents",
    description=("Search the project's documents and return the most relevant "
                 "passages with source and page. Use for any question about "
                 "the content of those documents."),
    parameters={"type": "object",
                "properties": {"query": {"type": "string",
                                         "description": "Search terms, not the raw question"}},
                "required": ["query"]})


# ── YOUR TOOL ─────────────────────────────────────────────────────────
def my_tool(argument: str) -> str:
    """TODO: one line saying what this does."""
    # ← TODO: implement. Return json.dumps({...}). On failure, return an
    #   error the MODEL can read and recover from, listing valid inputs.
    return json.dumps({"error": "not implemented"})


my_decl = types.FunctionDeclaration(
    name="my_tool",
    description="TODO: what it does, and when the model should use it",   # ← TODO
    parameters={"type": "object",
                "properties": {"argument": {"type": "string", "description": "TODO"}},
                "required": ["argument"]})

TOOLS = {"search_documents": search_documents, "my_tool": my_tool}
tool_config = types.Tool(function_declarations=[search_decl, my_decl])
CFG = types.GenerateContentConfig(tools=[tool_config], temperature=0.0)

## 7 · Agent loop

**Given complete.** Note the step cap and the allow-list — both are guardrails, not optional extras.

If your project does not need multi-step behaviour, use `answer()` from section 5 instead and say so in your presentation. A justified simple architecture scores better than an unjustified complex one.

In [ ]:
def run_agent(goal, max_steps=MAX_STEPS, verbose=True):
    history = [types.Content(role="user", parts=[types.Part.from_text(text=goal)])]
    trace = []

    for step in range(max_steps):
        r = client.models.generate_content(model=MODEL, contents=history, config=CFG)
        part = r.candidates[0].content.parts[0]

        if not getattr(part, "function_call", None):
            return {"answer": r.text, "trace": trace, "steps": step}

        call = part.function_call
        fn = TOOLS.get(call.name)                       # allow-list
        result = fn(**dict(call.args)) if fn else json.dumps(
            {"error": f"Unknown tool '{call.name}'", "available": list(TOOLS)})

        trace.append({"step": step, "tool": call.name, "args": dict(call.args)})
        if verbose:
            print(f"  step {step}: {call.name}({dict(call.args)})")

        history.append(r.candidates[0].content)
        history.append(types.Content(role="user", parts=[
            types.Part.from_function_response(name=call.name,
                                              response={"result": result})]))

    return {"answer": "Stopped: step limit reached.", "trace": trace, "steps": max_steps}

## 8 · Guardrails

One example, then a slot. Safety and reliability is 10 rubric points, and it only needs **one guardrail demonstrated** plus handling a failure without crashing.

In [ ]:
BANNED_OUTPUT = ["system prompt", "TODO: add anything your system must never say"]


def validate_output(text):
    """Example guardrail: block leaked instructions and off-topic answers."""
    for banned in BANNED_OUTPUT:
        if banned.lower() in text.lower():
            return "I cannot answer that."
    return text


def my_guardrail(question, text):
    """TODO: your own guardrail."""
    # ← TODO: implement ONE of:
    #   * refuse questions outside your project's scope
    #   * require a citation to be present in the answer
    #   * strip anything that looks like personal data
    return text


def safe_answer(question):
    """The public entry point. Never raises."""
    try:
        raw = answer(question)
        return my_guardrail(question, validate_output(raw))
    except Exception as e:
        # Handling a failure without crashing is worth rubric points.
        return f"Sorry - I could not answer that just now. ({type(e).__name__})"

## 9 · Instrumentation

From Day 4. Fill in the cost formula from the current pricing page — the numbers change every few months.

In [ ]:
LOG = []

PRICE_PER_1M_INPUT = 0.0        # ← TODO: real figure
PRICE_PER_1M_OUTPUT = 0.0       # ← TODO: real figure


def cost_of(prompt_tokens, output_tokens):
    # ← TODO (1 line): dollars for this request. Prices are per 1,000,000.
    return 0.0


def logged_answer(question):
    t0 = time.time()
    text = safe_answer(question)
    latency = time.time() - t0

    LOG.append({"timestamp": datetime.now().isoformat(timespec="seconds"),
                "question": question[:60],
                "latency": round(latency, 3),
                "chars_out": len(text)})
    return text


def metrics():
    df = pd.DataFrame(LOG)
    if df.empty:
        return "No requests logged yet."
    return df[["latency", "chars_out"]].describe()

## 10 · `demo()`

This is the function you run on Thursday, in front of the room. Rehearse it. Four minutes goes quickly.

In [ ]:
DEMO_QUESTIONS = [
    "TODO: a question that shows retrieval working well",       # ← TODO
    "TODO: a question that needs your tool",                    # ← TODO
    "TODO: a question your system correctly refuses or cannot answer",  # ← TODO
]


def demo():
    print("=" * 70)
    print(PROJECT_NAME)
    print("=" * 70)

    for q in DEMO_QUESTIONS:
        print()
        print("Q:", q)
        print("-" * 70)
        print(logged_answer(q))

    print()
    print("=" * 70)
    print("Measured metrics")
    print(metrics())


# demo()

## 11 · Golden set

Five questions you know the answers to. This is how you prove it works rather than claiming it — and it is what to show when someone asks on Thursday.

In [ ]:
GOLDEN = [
    {"q": "TODO: question 1", "must_contain": "TODO: a phrase that must appear"},  # ← TODO
    {"q": "TODO: question 2", "must_contain": "TODO"},                             # ← TODO
    {"q": "TODO: question 3", "must_contain": "TODO"},                             # ← TODO
    {"q": "TODO: question 4 — use an identifier or exact name", "must_contain": "TODO"},
    {"q": "TODO: question 5 — one whose answer is NOT in your documents",
     "must_contain": "do not know"},
]


def evaluate(search_fn=hybrid_search, k=TOP_K):
    hits = 0
    for item in GOLDEN:
        got = " ".join(r["text"] for r in search_fn(item["q"], k))
        ok = item["must_contain"].lower() in got.lower()
        hits += ok
        print(("PASS  " if ok else "MISS  ") + item["q"][:60])
    print()
    print(f"Hit rate: {hits}/{len(GOLDEN)}")
    return hits / len(GOLDEN)


# evaluate()

## 12 · README template

Copy everything in the block below into `README.md` in your repository, then
replace every `TODO`. All six SDAIA requirements are covered.

Wednesday's README clinic is where you write this, with me checking. READMEs
written the night before are always bad.

---

```markdown
# TODO: Project name

TODO: One sentence on what this does and who it is for.

## The problem

TODO: Two or three sentences. What was hard before this existed?

## Architecture

TODO: RAG, agent, or hybrid — and **why**, in one sentence tied to your use
case. Wrong-but-justified beats right-but-unexplained.

```
documents → chunking → embeddings → vector store → hybrid retrieval → model → answer with citations
```

TODO: Adjust that diagram to what you actually built.

## How to run it

1. Open `project_scaffold.ipynb` in Google Colab.
2. Add your `GEMINI_API_KEY` to Colab Secrets (key icon, left sidebar).
3. Upload your documents to the `my_docs` folder.
4. Run all cells, then call `demo()`.

## How to use it

TODO: Two or three example questions and the answers you get back.

## What works, and what does not

TODO: Be honest. Retrieval hit rate on your golden set: __/5.
TODO: One known limitation.

## Technical notes

- Model: `gemini-2.5-flash-lite`
- Embeddings: `gemini-embedding-001`
- Vector store: Chroma
- Retrieval: hybrid (BM25 + vector), alpha = TODO
- Chunking: TODO tokens, TODO overlap
- Guardrails: TODO

## Training programme

Built during **Applied Generative AI**, SDAIA Academy, Riyadh,
2–6 August 2026.

More SDAIA Academy projects: https://github.com/SDAIAAcademy

## Authors

TODO: names
```

---

### Repository checklist — all six SDAIA requirements

- [ ] Clear, comprehensive project description
- [ ] Professional `README.md`: the idea, how to run it, how to use it
- [ ] Appropriate technical documentation
- [ ] Real git history — several meaningful commits, not one dump
- [ ] The training programme named
- [ ] Link to https://github.com/SDAIAAcademy

## Reflection

Fill these in before you close the notebook. This is what I check when I come round.

**Which architecture did you choose — RAG, agent, or hybrid — and why, in one sentence?**

> _your answer here_

**What is your golden set hit rate, and which question fails?**

> _your answer here_

**Which guardrail did you implement, and what does it stop?**

> _your answer here_

**What broke, and what did you do about it? (This is worth marks on Thursday.)**

> _your answer here_

## If this breaks

The three most likely failures, and what to do about each.

| Symptom | Cause | Fix |
|---|---|---|
| `FileNotFoundError: my_docs` | No documents uploaded yet | Folder icon in the Colab sidebar → create `my_docs` → upload your files |
| Everything retrieves the same chunk | Your corpus is too small, or chunks are too large | Aim for at least 20 chunks; drop `CHUNK_SIZE` to 300 and re-index |
| `429` during indexing | Free-tier embedding rate limit | Raise `pause` in `embed()` to 2.0. Index once, then avoid re-running section 4 |